# Limpieza y Procesamiento de CSV - Cámaras Multiobjeto

Este notebook procesa los archivos CSV de las cámaras de tráfico y genera un único CSV consolidado con todos los registros.

## Estructura del CSV consolidado:
- **Cámara**: Nombre de la cámara
- **Mes**: Mes del registro
- **Año**: Año del registro
- **Día**: Día del mes
- **Fecha**: Fecha completa (DD/MM/AAAA)
- **Tipo**: Personas / Vehículos a motor / Vehículos sin motor
- **Subtipo**: Para vehículos a motor (Autobuses, Monovolúmenes, Sedanes pequeños, Camiones)
- **Dirección**: Avanzar / Retroceso
- **Valor**: Cantidad registrada
- **Latitud**: Coordenada geográfica
- **Longitud**: Coordenada geográfica

## 1. Importar Librerías

In [1]:
import os
import pandas as pd
import re
from datetime import datetime

## 2. Configurar Directorios

In [2]:
# Directorio base donde están los CSV originales
directorio_base = r'c:\Users\touri\Desktop\PB Peñíscola\Camaras_Multiobjeto\CSV'

# Directorio donde se guardarán los CSV limpios
directorio_salida = r'c:\Users\touri\Desktop\PB Peñíscola\Camaras_Multiobjeto\CSV_Limpios'

# Crear directorio de salida si no existe
os.makedirs(directorio_salida, exist_ok=True)

print(f"📁 Directorio de entrada: {directorio_base}")
print(f"📁 Directorio de salida: {directorio_salida}")

📁 Directorio de entrada: c:\Users\touri\Desktop\PB Peñíscola\Camaras_Multiobjeto\CSV
📁 Directorio de salida: c:\Users\touri\Desktop\PB Peñíscola\Camaras_Multiobjeto\CSV_Limpios


## 3. Coordenadas de las Cámaras

Define las coordenadas geográficas (latitud, longitud) de cada cámara.

In [3]:
# Diccionario de coordenadas de las cámaras
# Formato: 'Nombre Cámara': {'latitud': XX.XXXXX, 'longitud': X.XXXXX}

coordenadas_camaras = {
    'Avda. Akra Leuca': {
        'latitud': 40.358167,
        'longitud': 0.404143
    },
    'Avda. Del Mar - Marcelino Roca': {
        'latitud': 40.359441,
        'longitud': 0.404653
    },
    'Avda. España -Pza. Constitución': {
        'latitud': 40.358780,
        'longitud': 0.399557
    },
    'Avda. Papa Luna  -C.Navarra': {
        'latitud': 40.377994,
        'longitud': 0.407365
    },
    'Ayuntamiento': {
        'latitud': 40.358531,
        'longitud': 0.406687
    },
    'C Mayor - Baluarte Del Principe': {
        'latitud': 40.358257,
        'longitud': 0.407102
    },
    'C. Saiz De Carlos': {
        'latitud': 40.357431,
        'longitud': 0.406316
    },
    'Calle Fulladosa': {
        'latitud': 40.357128,
        'longitud': 0.407735
    },
    'Calle LLevant': {
        'latitud': 40.357945,
        'longitud': 0.404458
    },
    'Casco Antiguo - Marcelino Roca': {
        'latitud': 40.357979,
        'longitud': 0.406061
    },
    'Cruce CV-140 - Cami Vell': {
        'latitud': 40.374744,
        'longitud': 0.396649
    },
    'Estibaliz - Avda. Estación': {
        'latitud': 40.367798,
        'longitud': 0.393541
    },
    'Estibaliz - CV-140': {
        'latitud': 40.368122,
        'longitud': 0.393339
    },
    'Plaza Illueca - Avda- Papa Luna': {
        'latitud': 40.361652,
        'longitud': 0.401604
    },
    'Rotonda 3 CV-140': {
        'latitud': 40.388189,
        'longitud': 0.403056
    },
    # Agregar más cámaras aquí según vayas proporcionando las coordenadas
    # 'Nombre Cámara 16': {
    #     'latitud': XX.XXXXX,
    #     'longitud': X.XXXXX
    # },
}

print("📍 Coordenadas de cámaras configuradas:")
for camara, coords in coordenadas_camaras.items():
    print(f"   • {camara}: ({coords['latitud']}, {coords['longitud']})")

📍 Coordenadas de cámaras configuradas:
   • Avda. Akra Leuca: (40.358167, 0.404143)
   • Avda. Del Mar - Marcelino Roca: (40.359441, 0.404653)
   • Avda. España -Pza. Constitución: (40.35878, 0.399557)
   • Avda. Papa Luna  -C.Navarra: (40.377994, 0.407365)
   • Ayuntamiento: (40.358531, 0.406687)
   • C Mayor - Baluarte Del Principe: (40.358257, 0.407102)
   • C. Saiz De Carlos: (40.357431, 0.406316)
   • Calle Fulladosa: (40.357128, 0.407735)
   • Calle LLevant: (40.357945, 0.404458)
   • Casco Antiguo - Marcelino Roca: (40.357979, 0.406061)
   • Cruce CV-140 - Cami Vell: (40.374744, 0.396649)
   • Estibaliz - Avda. Estación: (40.367798, 0.393541)
   • Estibaliz - CV-140: (40.368122, 0.393339)
   • Plaza Illueca - Avda- Papa Luna: (40.361652, 0.401604)
   • Rotonda 3 CV-140: (40.388189, 0.403056)


## 5. Función para Procesar CSV

In [4]:
def procesar_csv(ruta_archivo, nombre_camara):
    """
    Procesa un archivo CSV y extrae los datos en formato limpio
    
    Args:
        ruta_archivo: Ruta completa al archivo CSV
        nombre_camara: Nombre de la cámara
    
    Returns:
        Lista de listas con los datos procesados
    """
    datos_limpios = []
    
    # Leer el archivo completo
    with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
        contenido = f.read()
    
    lineas = contenido.split('\n')
    
    # Procesar la primera tabla (Personas totales, Vehículos a motor, Vehículos sin motor)
    en_tabla_principal = False
    saltar_cabeceras = 0
    for i, linea in enumerate(lineas):
        # Buscar la línea que tiene "Hora" y "Personas" (puede tener N.º o N.? por encoding)
        if 'Hora' in linea and 'Personas' in linea and 'Veh' in linea:
            en_tabla_principal = True
            saltar_cabeceras = 2  # Saltar las 2 líneas de subcabecera (Avanzar/Retroceso y Valor/Tasa)
            continue
        
        if en_tabla_principal:
            # Saltar las líneas de cabecera después de encontrar la tabla
            if saltar_cabeceras > 0:
                saltar_cabeceras -= 1
                continue
            
            if 'Exportar contenido:Subtipos' in linea:
                break
            
            # Saltar líneas vacías
            if linea.strip() == '':
                continue
            
            partes = linea.split(';')
            if len(partes) < 3:
                continue
            
            try:
                num_linea = partes[0].strip()
                if not num_linea.isdigit():
                    continue
                
                # Extraer la fecha
                fecha_str = partes[1].strip() if len(partes) > 1 else ''
                if not fecha_str or 'Hora' in fecha_str:
                    continue
                
                match_fecha = re.search(r'(\d{4})/(\d{2})/(\d{2})', fecha_str)
                if not match_fecha:
                    continue
                
                year = match_fecha.group(1)
                month = match_fecha.group(2)
                day = match_fecha.group(3)
                fecha_completa = f"{year}/{month}/{day}"
                
                if len(partes) >= 20:
                    # Personas Avanzar
                    if partes[2].strip() and partes[2].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Personas', 'Personas', 'Avanzar', partes[2].strip()
                        ])
                    
                    # Personas Retroceso
                    if partes[5].strip() and partes[5].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Personas', 'Personas', 'Retroceso', partes[5].strip()
                        ])
                    
                   # NOTA: Vehículos a motor totales se omiten aquí
                    # Los subtipos se procesarán en la segunda tabla
                    
                    # Vehículos sin motor Avanzar
                    if partes[14].strip() and partes[14].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos sin motor', 'Vehículos sin motor', 'Avanzar', partes[14].strip()
                        ])
                    
                    # Vehículos sin motor Retroceso
                    if len(partes) > 17 and partes[17].strip() and partes[17].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos sin motor', 'Vehículos sin motor', 'Retroceso', partes[17].strip()
                        ])
            except (ValueError, IndexError):
                continue
    
    # Procesar la segunda tabla (Subtipos de vehículos a motor)
    en_tabla_subtipos = False
    saltar_cabeceras_subtipos = 0
    for i, linea in enumerate(lineas):
        if 'Exportar contenido:Subtipos' in linea:
            en_tabla_subtipos = True
            continue
        
        if en_tabla_subtipos:
            # Buscar el inicio de la tabla de subtipos (puede tener N.º o N.? por encoding)
            if 'Hora' in linea and 'Autobuses' in linea:
                saltar_cabeceras_subtipos = 2  # Saltar las 2 líneas de subcabecera (Avanzar/Retroceso y Valor/Tasa)
                continue
            
            # Saltar las líneas de cabecera
            if saltar_cabeceras_subtipos > 0:
                saltar_cabeceras_subtipos -= 1
                continue
            
            # Saltar líneas vacías
            if linea.strip() == '':
                continue
            
            partes = linea.split(';')
            if len(partes) < 3:
                continue
            
            try:
                num_linea = partes[0].strip()
                if not num_linea.isdigit():
                    continue
                
                fecha_str = partes[1].strip() if len(partes) > 1 else ''
                if not fecha_str or 'Hora' in fecha_str:
                    continue
                
                match_fecha = re.search(r'(\d{4})/(\d{2})/(\d{2})', fecha_str)
                if not match_fecha:
                    continue
                
                year = match_fecha.group(1)
                month = match_fecha.group(2)
                day = match_fecha.group(3)
                fecha_completa = f"{year}/{month}/{day}"
                
                if len(partes) >= 26:
                    # Autobuses Avanzar
                    if partes[2].strip() and partes[2].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Autobuses', 'Avanzar', partes[2].strip()
                        ])
                    
                    # Autobuses Retroceso
                    if partes[5].strip() and partes[5].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Autobuses', 'Retroceso', partes[5].strip()
                        ])
                    
                    # Monovolúmenes Avanzar
                    if partes[8].strip() and partes[8].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Monovolúmenes', 'Avanzar', partes[8].strip()
                        ])
                    
                    # Monovolúmenes Retroceso
                    if partes[11].strip() and partes[11].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Monovolúmenes', 'Retroceso', partes[11].strip()
                        ])
                    
                    # Sedanes pequeños Avanzar
                    if partes[14].strip() and partes[14].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Sedanes pequeños', 'Avanzar', partes[14].strip()
                        ])
                    
                    # Sedanes pequeños Retroceso
                    if partes[17].strip() and partes[17].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Sedanes pequeños', 'Retroceso', partes[17].strip()
                        ])
                    
                    # Camiones Avanzar
                    if partes[20].strip() and partes[20].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Camiones', 'Avanzar', partes[20].strip()
                        ])
                    
                    # Camiones Retroceso
                    if len(partes) > 23 and partes[23].strip() and partes[23].strip() != '--':
                        datos_limpios.append([
                            nombre_camara, month, year, day, fecha_completa,
                            'Vehículos a motor', 'Camiones', 'Retroceso', partes[23].strip()
                        ])
            except (ValueError, IndexError):
                continue
    
    # Ordenar los datos por fecha y tipo para que salgan juntos los del mismo día
    # Convertir a DataFrame temporal para ordenar
    import pandas as pd
    if datos_limpios:
        df_temp = pd.DataFrame(datos_limpios, columns=[
            'camara', 'mes', 'year', 'dia', 'fecha', 'tipo', 'subtipo', 'direccion', 'valor'
        ])
        # Ordenar por fecha, tipo, subtipo y dirección
        df_temp = df_temp.sort_values(['fecha', 'tipo', 'subtipo', 'direccion'])
        # Convertir de vuelta a lista de listas
        datos_limpios = df_temp.values.tolist()
    
    return datos_limpios

print("✅ Función de procesamiento definida")

✅ Función de procesamiento definida


## 6. Generar CSV Consolidado Único

Procesa todas las cámaras y genera un solo CSV con todos los registros.

In [5]:
# Lista para almacenar todos los datos
todos_los_datos = []

# Procesar cada carpeta de cámara
for nombre_camara_carpeta in os.listdir(directorio_base):
    ruta_camara = os.path.join(directorio_base, nombre_camara_carpeta)
    
    if not os.path.isdir(ruta_camara):
        continue
    
    # Extraer nombre limpio sin el número de carpeta
    # Ejemplo: "01 - Avda. Akra Leuca" -> "Avda. Akra Leuca"
    nombre_camara_limpio = re.sub(r'^\d+\s*-\s*', '', nombre_camara_carpeta)
    
    print(f"📹 Procesando: {nombre_camara_limpio}")
    
    # Procesar cada archivo CSV de esta cámara
    for archivo in os.listdir(ruta_camara):
        if archivo.endswith('.csv'):
            ruta_archivo = os.path.join(ruta_camara, archivo)
            datos = procesar_csv(ruta_archivo, nombre_camara_limpio)
            todos_los_datos.extend(datos)

# Crear DataFrame con todos los datos
df_consolidado = pd.DataFrame(todos_los_datos, columns=[
    'Cámara', 'Mes', 'Año', 'Día', 'Fecha', 
    'Tipo', 'Subtipo', 'Dirección', 'Valor'
])

# Agregar coordenadas según la cámara
def obtener_latitud(nombre_camara):
    coords = coordenadas_camaras.get(nombre_camara, {})
    return coords.get('latitud', None)

def obtener_longitud(nombre_camara):
    coords = coordenadas_camaras.get(nombre_camara, {})
    return coords.get('longitud', None)

df_consolidado['Latitud'] = df_consolidado['Cámara'].apply(obtener_latitud)
df_consolidado['Longitud'] = df_consolidado['Cámara'].apply(obtener_longitud)

# Convertir coordenadas a formato con punto decimal (en lugar de coma)
# Esto asegura que Power BI las reconozca correctamente como coordenadas
df_consolidado['Latitud'] = df_consolidado['Latitud'].apply(lambda x: f"{x:.6f}" if pd.notna(x) else "")
df_consolidado['Longitud'] = df_consolidado['Longitud'].apply(lambda x: f"{x:.6f}" if pd.notna(x) else "")

# Guardar CSV consolidado con separador punto y coma y decimales con punto
archivo_consolidado = os.path.join(directorio_salida, 'consolidado_todas_camaras.csv')
df_consolidado.to_csv(archivo_consolidado, index=False, encoding='utf-8-sig', sep=';', decimal='.')

print(f"\n✅ Archivo consolidado creado: {archivo_consolidado}")
print(f"📊 Total de registros: {len(df_consolidado)}")
print(f"📊 Total de cámaras: {df_consolidado['Cámara'].nunique()}")
print(f"\nPrimeras 15 filas:")
display(df_consolidado.head(15))

📹 Procesando: Avda. Akra Leuca
📹 Procesando: Avda. Del Mar - Marcelino Roca
📹 Procesando: Avda. España -Pza. Constitución
📹 Procesando: Avda. Del Mar - Marcelino Roca
📹 Procesando: Avda. España -Pza. Constitución
📹 Procesando: Avda. Papa Luna  -C.Navarra
📹 Procesando: Ayuntamiento
📹 Procesando: Avda. Papa Luna  -C.Navarra
📹 Procesando: Ayuntamiento
📹 Procesando: C Mayor - Baluarte Del Principe
📹 Procesando: C. Saiz De Carlos
📹 Procesando: C Mayor - Baluarte Del Principe
📹 Procesando: C. Saiz De Carlos
📹 Procesando: Calle Fulladosa
📹 Procesando: Calle LLevant
📹 Procesando: Calle Fulladosa
📹 Procesando: Calle LLevant
📹 Procesando: Casco Antiguo - Marcelino Roca
📹 Procesando: Cruce CV-140 - Cami Vell
📹 Procesando: Casco Antiguo - Marcelino Roca
📹 Procesando: Cruce CV-140 - Cami Vell
📹 Procesando: Estibaliz - Avda. Estación
📹 Procesando: Estibaliz - CV-140
📹 Procesando: Estibaliz - Avda. Estación
📹 Procesando: Estibaliz - CV-140
📹 Procesando: Plaza Illueca - Avda- Papa Luna
📹 Procesando: R

,Cámara,Mes,Año,Día,Fecha,Tipo,Subtipo,Dirección,Valor,Latitud,Longitud
0,Avda. Akra Leuca,05,2025,01,2025/05/01,Personas,Personas,Avanzar,2314,40.358167,0.404143
1,Avda. Akra Leuca,05,2025,01,2025/05/01,Personas,Personas,Retroceso,3492,40.358167,0.404143
2,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Autobuses,Avanzar,71,40.358167,0.404143
3,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Autobuses,Retroceso,57,40.358167,0.404143
4,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Camiones,Avanzar,18,40.358167,0.404143
5,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Camiones,Retroceso,14,40.358167,0.404143
6,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Monovolúmenes,Avanzar,218,40.358167,0.404143
7,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Monovolúmenes,Retroceso,164,40.358167,0.404143
8,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Sedanes pequeños,Avanzar,2734,40.358167,0.404143
9,Avda. Akra Leuca,05,2025,01,2025/05/01,Vehículos a motor,Sedanes pequeños,Retroceso,3213,40.358167,0.404143
